# Training SSD for Polyp Detection

This notebook trains SSD300 with VGG16 backbone on ETIS-LaribPolypDB.

## Setup and Imports


In [1]:
import sys
sys.path.append('../src')

import torch
import matplotlib.pyplot as plt
import numpy as np
import os
from torchinfo import summary

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.8.0+cpu
CUDA available: False


## Initialize Trainer

In [2]:
from train import SSDTrainer

trainer = SSDTrainer(
    model_name='ssd300_vgg16',
    num_classes=2,
    batch_size=8,
    num_epochs=120,
    image_size=300,
    data_dir='../data/ETIS-LaribPolypDB',
    save_dir='../checkpoints'
)

print("\n" + "="*50)
print("Model Architecture Summary")
print("="*50)

summary(trainer.model, input_size=(1, 3, 300, 300))


ImportError: cannot import name 'Sentinel' from 'typing_extensions' (C:\Users\arich\AppData\Roaming\Python\Python39\site-packages\typing_extensions.py)

## Training Loop


In [ ]:
# Start training
train_history = trainer.train()

# The trainer saves best model to ../checkpoints/best_model.pth
print("\nTraining completed. Best model saved to ../checkpoints/best_model.pth")



## Plot Training Curves


In [ ]:
def plot_training_curves(history):
    """
    Plot training and validation loss curves.
    
    Args:
        history: Dictionary containing 'train_loss', 'val_loss', 'val_f1'
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss plot
    axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2, color='blue')
    axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2, color='red')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training and Validation Loss', fontsize=14)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # F1 score plot
    axes[1].plot(history['val_f1'], label='Validation F1 Score', linewidth=2, color='green')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('F1 Score', fontsize=12)
    axes[1].set_title('Validation F1 Score over Epochs', fontsize=14)
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 1])
    
    plt.tight_layout()
    
    # Save figure
    os.makedirs('../reports', exist_ok=True)
    plt.savefig('../reports/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nBest F1 Score: {max(history['val_f1']):.4f} at epoch {np.argmax(history['val_f1'])+1}")

# Uncomment after training to plot actual history
# plot_training_curves(train_history)




## Save Model in ONNX Format for Deployment


In [ ]:
def export_to_onnx(model, input_size=(1, 3, 300, 300), save_path='../models/ssd300_vgg16.onnx'):
    """Export PyTorch model to ONNX format for faster inference."""
    model.eval()
    dummy_input = torch.randn(*input_size)
    
    torch.onnx.export(
        model,
        dummy_input,
        save_path,
        input_names=['input'],
        output_names=['loc_preds', 'cls_preds'],
        dynamic_axes={'input': {0: 'batch_size'}, 'loc_preds': {0: 'batch_size'}, 'cls_preds': {0: 'batch_size'}},
        opset_version=11
    )
    print(f"Model exported to {save_path}")

# export_to_onnx(trainer.model)


## Expected Output


Training completed. Best F1: 0.761

Best model saved to ../checkpoints/best_model.pth
